In [ ]:
def compute_hours_since_known_rain(ppt_series):
    """
    Computes HoursSinceRain with a hybrid strategy.
    Dynamically adjusts max_tolerated_nans based on ppt NaN density.
    """

    # Step 1: Compute NaN density
    nan_ratio = ppt_series.isna().mean()

    # Step 2: Decide tolerance level
    if nan_ratio < 0.01:
        max_tolerated_nans = 0
    elif nan_ratio < 0.05:
        max_tolerated_nans = 1
    elif nan_ratio < 0.15:
        max_tolerated_nans = 2
    elif nan_ratio < 0.25:
        max_tolerated_nans = 3
    else:
        max_tolerated_nans = 5

    # Step 3: Apply hybrid logic
    hours_since = []
    count = None
    unknown_streak = 0

    for v in ppt_series:
        if pd.isna(v):
            unknown_streak += 1
            if count is None or unknown_streak > max_tolerated_nans:
                count = None  # We've lost confidence in last known rain
            hours_since.append(np.nan)
        elif v > 0:
            count = 0
            unknown_streak = 0
            hours_since.append(0)
        elif v == 0:
            unknown_streak = 0
            if count is None:
                hours_since.append(np.nan)
            else:
                count += 1
                hours_since.append(count)

    return hours_since
